# 查看与探索 Fraud Detection Parquet 数据

这个 Notebook 只读取训练集，完成 EDA 和训练集历史图特征分析。外部测试集留到模型和阈值确定后才用于最终推断。


In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path('.')
TRAIN_PATH = DATA_DIR / 'train_fraud.parquet'


In [2]:
train_df = pd.read_parquet(TRAIN_PATH)
print('训练集 train_fraud.parquet')
print(f'形状: {train_df.shape}')
display(train_df.head())


训练集 train_fraud.parquet
形状: (3144097, 11)


,hours_elapsed,transaction_type,transaction_amount,initiating_customer,initiater_balance_before,initiater_balance_after,recipient_customer,recipient_balance_before,recipient_balance_after,is_fraud,is_flagged_fraud
6281802,652,CASH_OUT,2911505.23,C-66cc56,2911505.23,0.0,C-8ba1aa,474059.17,3385564.40,1,0
6281804,652,CASH_OUT,740124.95,C-33af6f,740124.95,0.0,C-cc9f4c,1382863.66,2122988.62,1,0
6281808,652,CASH_OUT,92043.22,C-ab096b,92043.22,0.0,C-288948,3090723.25,3182766.47,1,0
6281816,653,CASH_OUT,119.65,C-e68290,119.65,0.0,C-030445,1183575.58,1183695.22,1,0
6281822,654,CASH_OUT,48712.12,C-897dab,48712.12,0.0,C-f8a5f6,258570.90,307283.02,1,0


In [3]:
print('训练集列名和数据类型')
display(train_df.dtypes.to_frame('dtype'))


训练集列名和数据类型


,dtype
hours_elapsed,int64
transaction_type,object
transaction_amount,float64
initiating_customer,object
initiater_balance_before,float64
initiater_balance_after,float64
recipient_customer,object
recipient_balance_before,float64
recipient_balance_after,float64
is_fraud,int64


In [4]:
# 账户 ID 前缀的语义见 columns.txt。这里只使用训练集。
for role in ('initiating_customer', 'recipient_customer'):
    train_df[f'{role}_is_C'] = train_df[role].astype('string').str.startswith('C').astype('int8')
    train_df[f'{role}_is_M'] = train_df[role].astype('string').str.startswith('M').astype('int8')

new_columns = [
    'initiating_customer_is_C', 'initiating_customer_is_M',
    'recipient_customer_is_C', 'recipient_customer_is_M',
]
display(train_df[['initiating_customer', 'recipient_customer'] + new_columns].head())


,initiating_customer,recipient_customer,initiating_customer_is_C,initiating_customer_is_M,recipient_customer_is_C,recipient_customer_is_M
6281802,C-66cc56,C-8ba1aa,1,0,1,0
6281804,C-33af6f,C-cc9f4c,1,0,1,0
6281808,C-ab096b,C-288948,1,0,1,0
6281816,C-e68290,C-030445,1,0,1,0
6281822,C-897dab,C-f8a5f6,1,0,1,0


In [5]:
train_df

,hours_elapsed,transaction_type,transaction_amount,initiating_customer,initiater_balance_before,initiater_balance_after,recipient_customer,recipient_balance_before,recipient_balance_after,is_fraud,is_flagged_fraud,initiating_customer_is_C,initiating_customer_is_M,recipient_customer_is_C,recipient_customer_is_M
6281802,652,CASH_OUT,2911505.23,C-66cc56,2911505.23,0.0,C-8ba1aa,474059.17,3385564.40,1,0,1,0,1,0
6281804,652,CASH_OUT,740124.95,C-33af6f,740124.95,0.0,C-cc9f4c,1382863.66,2122988.62,1,0,1,0,1,0
6281808,652,CASH_OUT,92043.22,C-ab096b,92043.22,0.0,C-288948,3090723.25,3182766.47,1,0,1,0,1,0
6281816,653,CASH_OUT,119.65,C-e68290,119.65,0.0,C-030445,1183575.58,1183695.22,1,0,1,0,1,0
6281822,654,CASH_OUT,48712.12,C-897dab,48712.12,0.0,C-f8a5f6,258570.90,307283.02,1,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6281794,651,CASH_OUT,9996886.64,C-bfbf7f,9996886.64,0.0,C-562c48,20429.76,10017316.40,1,0,1,0,1,0
6281795,651,TRANSFER,595558.89,C-074ed3,595558.89,0.0,C-820761,0.00,0.00,1,0,1,0,1,0
6281796,651,CASH_OUT,595558.89,C-0bca72,595558.89,0.0,C-ead483,5346086.17,5941645.06,1,0,1,0,1,0
6281797,651,TRANSFER,65494.78,C-e91c87,65494.78,0.0,C-538470,0.00,0.00,1,0,1,0,1,0


In [6]:
# 外部测试集只在最终预测阶段读取；这里不展示或使用它。


In [7]:
# 为时间 EDA 排序；同一小时内 ID 的顺序不代表交易真实先后。
train_df = train_df.sort_values(
    ['hours_elapsed', 'initiating_customer'], kind='stable'
)
display(train_df.head())


,hours_elapsed,transaction_type,transaction_amount,initiating_customer,initiater_balance_before,initiater_balance_after,recipient_customer,recipient_balance_before,recipient_balance_after,is_fraud,is_flagged_fraud,initiating_customer_is_C,initiating_customer_is_M,recipient_customer_is_C,recipient_customer_is_M
682,1,PAYMENT,154.87,C-00121c,9339.00,9184.13,M-6404bd,0.00,0.00,0,0,1,0,0,1
417,1,CASH_IN,104605.78,C-0018b7,4968145.50,5072751.28,C-7f285c,207302.14,92307.65,0,0,1,0,1,0
15,1,CASH_OUT,229133.94,C-004790,15325.00,0.00,C-c71f1b,5083.00,51513.44,0,0,1,0,1,0
2518,1,CASH_IN,231460.09,C-007883,3411151.57,3642611.66,C-8e166f,10748319.81,10155564.02,0,0,1,0,1,0
2500,1,CASH_IN,132540.87,C-009549,38203.34,170744.21,C-0f6339,507973.21,84916.82,0,0,1,0,1,0


In [8]:
# EDA helper functions are kept in code/plot to keep this Notebook readable.
import sys
from pathlib import Path

PLOT_CODE_DIR = Path.cwd() / 'code' / 'plot'
if str(PLOT_CODE_DIR) not in sys.path:
    sys.path.append(str(PLOT_CODE_DIR))

from eda_plots import (
    get_dataset_quality_metrics,
    plot_label_distribution,
    plot_time_distribution,
    summarize_data_quality,
)

# Data-quality summary for the training data used in EDA/model development.
quality_metrics = get_dataset_quality_metrics(train_df)
print('Training-data quality metrics:')
display(pd.Series(quality_metrics, name='value'))
display(summarize_data_quality(train_df))

# Generate the two requested EDA figures. The external test set is not used here.
label_figure = plot_label_distribution(train_df, output_dir='image/数据质量和标签分布')
time_figure = plot_time_distribution(train_df, output_dir='image/时间分布')
print(f'Saved label figure: {label_figure}')
print(f'Saved time figure: {time_figure}')

Training-data quality metrics:


rows                                                          3144097
columns                                                            15
missing_cells                                                       0
duplicate_rows                                                      0
constant_columns    [initiating_customer_is_C, initiating_customer...
Name: value, dtype: object

,dtype,missing_count,missing_rate_pct,unique_values
hours_elapsed,int64,0,0.0,481
transaction_type,object,0,0.0,5
transaction_amount,float64,0,0.0,2960195
initiating_customer,object,0,0.0,2760260
initiater_balance_before,float64,0,0.0,1065893
initiater_balance_after,float64,0,0.0,1312667
recipient_customer,object,0,0.0,785435
recipient_balance_before,float64,0,0.0,2302623
recipient_balance_after,float64,0,0.0,2105417
is_fraud,int64,0,0.0,2


Saved label figure: image/数据质量和标签分布/label_distribution.pdf
Saved time figure: image/时间分布/time_distribution_training.pdf


### Additional data-quality diagnostics

These two diagnostics use the training data only. The zero-value plot distinguishes observed zeros from missing values, while the cardinality plot shows which raw fields are high-cardinality and therefore need special handling. The four customer-type indicator columns are excluded from the cardinality plot because they were engineered earlier.

In [9]:
from data_quality_eda import (
    plot_field_cardinality,
    plot_zero_value_rates,
    summarize_field_cardinality,
    summarize_zero_rates,
)
from IPython.display import IFrame

zero_value_summary = summarize_zero_rates(train_df)
field_cardinality_summary = summarize_field_cardinality(train_df, engineered_columns=new_columns)

print('Exact-zero summary by label:')
display(zero_value_summary)
print('Raw-field cardinality summary:')
display(field_cardinality_summary)

zero_value_figure = plot_zero_value_rates(
    train_df,
    output_dir='image/数据质量和标签分布',
)
field_cardinality_figure = plot_field_cardinality(
    train_df,
    output_dir='image/数据质量和标签分布',
    engineered_columns=new_columns,
)
print(f'Saved: {zero_value_figure}')
print(f'Saved: {field_cardinality_figure}')

# Preview the PDF figures directly in the notebook; no PNG files are created.
display(IFrame(src=str(zero_value_figure), width='100%', height=520))
display(IFrame(src=str(field_cardinality_figure), width='100%', height=650))

Exact-zero summary by label:


,label,label_name,column,zero_count,row_count,zero_rate_pct
0,0,Legitimate,transaction_amount,0,3143740,0.000000
1,0,Legitimate,initiater_balance_before,1098676,3143740,34.948056
2,0,Legitimate,initiater_balance_after,1821150,3143740,57.929409
3,0,Legitimate,recipient_balance_before,824437,3143740,26.224720
4,0,Legitimate,recipient_balance_after,746304,3143740,23.739368
5,1,Fraud,transaction_amount,3,357,0.840336
6,1,Fraud,initiater_balance_before,3,357,0.840336
7,1,Fraud,initiater_balance_after,349,357,97.759104
8,1,Fraud,recipient_balance_before,160,357,44.817927
9,1,Fraud,recipient_balance_after,109,357,30.532213


Raw-field cardinality summary:


,column,role,dtype,unique_count,unique_rate_pct,missing_count
0,transaction_amount,Numeric,float64,2960195,94.150880,0
1,initiating_customer,Customer ID,object,2760260,87.791821,0
2,recipient_balance_before,Numeric,float64,2302623,73.236386,0
3,recipient_balance_after,Numeric,float64,2105417,66.964124,0
4,initiater_balance_after,Numeric,float64,1312667,41.750207,0
5,initiater_balance_before,Numeric,float64,1065893,33.901403,0
6,recipient_customer,Customer ID,object,785435,24.981259,0
7,hours_elapsed,Time,int64,481,0.015299,0
8,transaction_type,Category,object,5,0.000159,0
9,is_flagged_fraud,System flag,int64,2,0.000064,0


Saved: image/数据质量和标签分布/zero_value_rates.pdf
Saved: image/数据质量和标签分布/field_cardinality.pdf


## Temporal distribution shift within the training data

This section compares three equal-width `hours_elapsed` windows in the training data. It examines transaction-type composition, transaction volume, fraud prevalence, and numeric distributions. The external test set is not used for these EDA calculations.
The following rule-stability plot separates rule hits, missed fraud and legitimate transaction volume in explicit hour windows.


In [10]:
from distribution_shift_eda import (
    plot_numeric_distribution_shift,
    plot_temporal_composition_shift,
    summarize_temporal_shift,
)
from IPython.display import IFrame

distribution_shift = summarize_temporal_shift(train_df)
print('Temporal-window summary:')
display(distribution_shift['period_summary'])
print('Transaction-type composition by temporal window (%):')
display(distribution_shift['composition_pct'])
print(f"Early–Late transaction-type Jensen–Shannon distance: {distribution_shift['type_js_distance_early_late']:.4f}")
print('Numeric distribution-shift summary:')
display(distribution_shift['numeric_summary'])

temporal_composition_figure = plot_temporal_composition_shift(
    train_df,
    output_dir='image/分布偏移',
)
numeric_distribution_figure = plot_numeric_distribution_shift(
    train_df,
    output_dir='image/分布偏移',
)
print(f'Saved: {temporal_composition_figure}')
print(f'Saved: {numeric_distribution_figure}')
display(IFrame(src=str(temporal_composition_figure), width='100%', height=520))
display(IFrame(src=str(numeric_distribution_figure), width='100%', height=680))

Temporal-window summary:


,transaction_count,fraud_count,fraud_rate_pct,start_hour,end_hour
time_period,,,,,
Early,2244559,0,0.000000,1,242
Middle,828876,0,0.000000,249,496
Late,70662,357,0.505222,500,743


Transaction-type composition by temporal window (%):


transaction_type,CASH_IN,CASH_OUT,DEBIT,PAYMENT,TRANSFER
time_period,,,,,
Early,25.083814,40.236055,0.739254,24.261202,9.679674
Middle,30.419870,46.783114,0.882762,8.630965,13.283290
Late,31.647845,41.465002,1.093940,11.356882,14.436331


Early–Late transaction-type Jensen–Shannon distance: 0.1282
Numeric distribution-shift summary:


,column,early_median,late_median,ks_statistic,ks_p_value
0,transaction_amount,107565.78,141880.105,0.111205,0.000000e+00
1,initiater_balance_before,14708.00,20062.500,0.045601,3.142636e-124
2,initiater_balance_after,0.00,0.000,0.045987,2.454766e-126
3,recipient_balance_before,347910.51,1158881.975,0.218557,0.000000e+00
4,recipient_balance_after,451149.39,1277733.630,0.203842,0.000000e+00


Saved: image/分布偏移/temporal_composition_shift.pdf
Saved: image/分布偏移/numeric_distribution_shift.pdf


In [ ]:
# Pre-transaction full-balance rule, shown by transaction type and hour window.
from full_balance_eda import plot_rule_over_time, summarize_rule_over_time

rule_time_summary = summarize_rule_over_time(train_df)
display(rule_time_summary)
rule_time_figure = plot_rule_over_time(train_df)
print(f'Saved: {rule_time_figure}')


## 1. Transaction type and fraud relationship

This block compares transaction volume, fraud count, and fraud rate by transaction type. It uses the training data only.

In [11]:
from transaction_type_eda import (
    plot_transaction_type_counts,
    plot_transaction_type_fraud_rates,
    summarize_transaction_types,
)

transaction_type_summary = summarize_transaction_types(train_df)
display(transaction_type_summary)

transaction_type_count_figure = plot_transaction_type_counts(train_df)
transaction_type_rate_figure = plot_transaction_type_fraud_rates(train_df)
print(f'Saved: {transaction_type_count_figure}')
print(f'Saved: {transaction_type_rate_figure}')

,transaction_count,fraud_count,flagged_count,fraud_rate_pct
transaction_type,,,,
CASH_IN,837527,0,0,0.000000
CASH_OUT,1320196,250,0,0.018937
DEBIT,24683,0,0,0.000000
PAYMENT,624122,0,0,0.000000
TRANSFER,337569,107,2,0.031697


Saved: image/交易类型与欺诈的关系/transaction_type_counts.pdf
Saved: image/交易类型与欺诈的关系/transaction_type_fraud_rates.pdf


## 2. Transaction amount and balance behavior

The amount distribution uses log1p(transaction_amount). Balance diagnostics are summarized separately by transaction type because the expected money flow differs across transaction types.
The following two plots isolate the positive-balance full-withdrawal rule and display amount-to-starting-balance bins with support counts and fraud rates.


In [12]:
from amount_balance_eda import (
    plot_amount_balance_behavior,
    summarize_amounts,
    summarize_balance_behavior,
)

amount_summary = summarize_amounts(train_df)
balance_summary = summarize_balance_behavior(train_df)
display(amount_summary)
display(balance_summary['by_transaction_type'])

amount_balance_figure = plot_amount_balance_behavior(train_df)
print(f'Saved: {amount_balance_figure}')

,transaction_count,mean,median,q01,q25,q75,q99,zero_amount_rate_pct
Legitimate,3143740,2.171483e+05,121039.305,669.14,27075.0775,250276.795,1.783519e+06,0.000000
Fraud,357,1.523691e+06,349627.640,1008.79,91882.6000,1411271.230,1.000000e+07,0.840336


,median_amount_to_initiator_before_pct,initiator_after_zero_rate_pct,recipient_after_zero_rate_pct,initiator_balance_consistency_pct,recipient_balance_consistency_pct
CASH_IN,8.900811,0.000000,13.408881,0.000000,0.000000
CASH_OUT,467.329224,89.410057,0.657705,10.084715,76.084990
DEBIT,9.947570,28.703966,1.438237,69.525584,77.194830
PAYMENT,19.023533,49.504744,100.000000,46.993537,0.000000
TRANSFER,1474.511050,96.293202,0.281424,3.478104,75.047768


Saved: image/交易金额与余额行为/amount_balance_behavior.pdf


In [ ]:
# Amount-to-starting-balance relationship; zero denominators stay separate.
# These are full-training-set EDA findings, not held-out model scores.
from full_balance_eda import (
    plot_balance_ratio_bins,
    plot_full_balance_rule,
    summarize_balance_ratio_bins,
    summarize_full_balance_rule,
)

full_balance_summary = summarize_full_balance_rule(train_df)
ratio_bin_summary = summarize_balance_ratio_bins(train_df)
display(full_balance_summary)
display(ratio_bin_summary)
full_balance_figure = plot_full_balance_rule(train_df)
ratio_bin_figure = plot_balance_ratio_bins(train_df)
print(f'Saved: {full_balance_figure}')
print(f'Saved: {ratio_bin_figure}')


## 3. Customer types and account history

The role-pair analysis uses the four binary customer-type features created earlier. Historical coverage is evaluated with strictly earlier hours; transactions in the same hour are not treated as ordered.

In [13]:
from customer_history_eda import plot_customer_history, summarize_customer_history

customer_history = summarize_customer_history(train_df)
display(customer_history['role_summary'])
display(customer_history['history_metrics'])

print('Initiating-account activity summary:')
display(customer_history['initiator_activity'].describe(percentiles=[0.5, 0.75, 0.9, 0.99]).T)
print('Recipient-account activity summary:')
display(customer_history['recipient_activity'].describe(percentiles=[0.5, 0.75, 0.9, 0.99]).T)

customer_history_figure = plot_customer_history(train_df)
print(f'Saved: {customer_history_figure}')

,transaction_count,fraud_count,fraud_rate_pct
_role_pair,,,
C→C,2519975,357,0.014167
C→M,624122,0,0.000000
M→C,0,0,0.000000
M→M,0,0,0.000000


unique_initiating_accounts                          2760260
repeated_initiating_accounts                         339521
unique_recipient_accounts                            785435
repeated_recipient_accounts                          207889
accounts_in_both_roles                                67029
initiators_active_in_multiple_hours                  337901
recipients_active_in_multiple_hours                  207591
transactions_with_strict_prior_initiator_history     381928
transactions_with_strict_prior_recipient_history    2321919
Name: value, dtype: int64

Initiating-account activity summary:


,count,mean,std,min,50%,75%,90%,99%,max
transaction_count,2760260.0,1.139058,0.394123,1.0,1.00,1.0000,2.000,3.000000e+00,7.00
active_hours,2760260.0,1.138270,0.392785,1.0,1.00,1.0000,2.000,3.000000e+00,7.00
unique_recipients,2760260.0,1.139058,0.394122,1.0,1.00,1.0000,2.000,3.000000e+00,7.00
total_amount,2760260.0,247513.496864,662996.607677,0.0,140771.65,281205.1675,479051.043,1.956542e+06,74132470.87


Recipient-account activity summary:


,count,mean,std,min,50%,75%,90%,99%,max
transaction_count,785435.0,4.003001,7.545823e+00,1.0,1.00,2.000,14.000,3.600000e+01,1.130000e+02
active_hours,785435.0,3.662372,6.367320e+00,1.0,1.00,2.000,13.000,2.900000e+01,5.800000e+01
unique_initiators,785435.0,4.002998,7.545817e+00,1.0,1.00,2.000,14.000,3.600000e+01,1.130000e+02
total_amount,785435.0,869838.503319,3.090635e+06,0.0,15550.98,91831.965,2829675.662,1.229022e+07,2.993744e+08


Saved: image/客户类型与账户历史/customer_history.pdf


## 4. Existing fraud-flagging system

is_flagged_fraud is evaluated as a pre-existing binary classifier against the actual is_fraud label.

In [14]:
from system_flag_eda import plot_system_flag_performance, summarize_system_flag

system_flag_result = summarize_system_flag(train_df)
print('Confusion matrix:')
display(system_flag_result['confusion_matrix'])
print('Fraud-class metrics:')
display(system_flag_result['metrics'].to_frame())

system_flag_figure = plot_system_flag_performance(train_df)
print(f'Saved: {system_flag_figure}')

Confusion matrix:


is_flagged_fraud,0,1
is_fraud,,
0,3143740,0
1,355,2


Fraud-class metrics:


,value
precision,1.000000
recall,0.005602
f1_score,0.011142


Saved: image/已有系统标记的效果/system_flag_performance.pdf


## GFP-inspired historical graph features

We represent each recorded transaction as a directed initiator-to-recipient edge. The five lightweight graph statistics below count only transactions in **strictly earlier hours**. They are inspired by the vertex-statistics component of GFP; they are not Snap ML's full cycle/scatter-gather miner. No external test rows or fraud labels enter feature generation. The comparisons are descriptive training-set EDA, not validation performance.


In [15]:
from pathlib import Path
import sys

GFP_CODE_DIR = Path.cwd() / 'code' / 'GFP'
if str(GFP_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(GFP_CODE_DIR))

from graph_features import build_gfp_features
from gfp_eda import summarize_gfp_by_label, plot_gfp_feature

gfp_features = build_gfp_features(train_df)
assert len(gfp_features) == len(train_df)
assert gfp_features.index.equals(train_df.index)
print('Training-only graph feature preview:')
display(gfp_features.head())

gfp_summary = summarize_gfp_by_label(train_df, gfp_features)
metrics_dir = Path('outputs/metrics')
metrics_dir.mkdir(parents=True, exist_ok=True)
gfp_summary_path = metrics_dir / 'gfp_feature_summary.csv'
gfp_summary.to_csv(gfp_summary_path, index=False)

for feature in ('gfp_target_prior_in_fan', 'gfp_source_prior_out_fan'):
    print(feature)
    display(gfp_summary[gfp_summary['feature'].eq(feature)])
    pdf_path = plot_gfp_feature(
        gfp_summary, feature, output_dir='image/图拓扑特征'
    )
    print(f'Saved {pdf_path}')
# A time-matched descriptive check: fraud begins at hour 635, while the last
# legitimate record is at hour 718. This is EDA, not an evaluation split.
late_mask = train_df['hours_elapsed'].between(635, 718)
late_summary = summarize_gfp_by_label(
    train_df.loc[late_mask], gfp_features.loc[late_mask]
)
late_summary_path = metrics_dir / 'gfp_feature_summary_hours_635_718.csv'
late_summary.to_csv(late_summary_path, index=False)
late_pdf = plot_gfp_feature(
    late_summary,
    'gfp_target_prior_in_fan',
    output_dir='image/图拓扑特征',
    context='hours 635-718',
    filename='gfp_target_prior_in_fan_hours_635_718',
)
print(f'Saved {gfp_summary_path}, {late_summary_path}, and {late_pdf}')


Training-only graph feature preview:


,gfp_source_prior_out_degree,gfp_source_prior_out_fan,gfp_target_prior_in_degree,gfp_target_prior_in_fan,gfp_pair_prior_count
682,0,0,0,0,0
417,0,0,0,0,0
15,0,0,0,0,0
2518,0,0,0,0,0
2500,0,0,0,0,0


gfp_target_prior_in_fan


,feature,transaction_type,bucket,transaction_count,fraud_count,fraud_rate_pct
36,gfp_target_prior_in_fan,CASH_OUT,0,142958,28,0.019586
37,gfp_target_prior_in_fan,CASH_OUT,1,84902,47,0.055358
38,gfp_target_prior_in_fan,CASH_OUT,2,77236,19,0.024600
39,gfp_target_prior_in_fan,CASH_OUT,3–5,212108,28,0.013201
40,gfp_target_prior_in_fan,CASH_OUT,6–10,291767,33,0.011310
41,gfp_target_prior_in_fan,CASH_OUT,11+,511225,95,0.018583
42,gfp_target_prior_in_fan,TRANSFER,0,27240,98,0.359765
43,gfp_target_prior_in_fan,TRANSFER,1,15485,2,0.012916
44,gfp_target_prior_in_fan,TRANSFER,2,14263,0,0.000000
45,gfp_target_prior_in_fan,TRANSFER,3–5,42553,3,0.007050


Saved image/图拓扑特征/gfp_target_prior_in_fan.pdf
gfp_source_prior_out_fan


,feature,transaction_type,bucket,transaction_count,fraud_count,fraud_rate_pct
12,gfp_source_prior_out_fan,CASH_OUT,0,1167609,191,0.016358
13,gfp_source_prior_out_fan,CASH_OUT,1,136210,53,0.038911
14,gfp_source_prior_out_fan,CASH_OUT,2,14790,4,0.027045
15,gfp_source_prior_out_fan,CASH_OUT,3–5,1587,2,0.126024
16,gfp_source_prior_out_fan,CASH_OUT,6–10,0,0,NaN
17,gfp_source_prior_out_fan,CASH_OUT,11+,0,0,NaN
18,gfp_source_prior_out_fan,TRANSFER,0,298695,68,0.022766
19,gfp_source_prior_out_fan,TRANSFER,1,34941,34,0.097307
20,gfp_source_prior_out_fan,TRANSFER,2,3556,5,0.140607
21,gfp_source_prior_out_fan,TRANSFER,3–5,377,0,0.000000


Saved image/图拓扑特征/gfp_source_prior_out_fan.pdf


Saved outputs/metrics/gfp_feature_summary.csv, outputs/metrics/gfp_feature_summary_hours_635_718.csv, and image/图拓扑特征/gfp_target_prior_in_fan_hours_635_718.pdf
